# Mixed-effects models of confidence change

Estimate confidence change by accuracy trajectory and compare each LLM arm with control.

## Load packages

In [ ]:
# Pakete laden — Installation war einmalig nötig (install.packages(...)).
# Diese Zelle jedes Mal ausführen; erneut installieren nur bei fehlendem Paket:
#   install.packages(c("readxl", "dplyr", "lme4", "emmeans"))

library(readxl)
library(dplyr)
library(lme4)
library(emmeans)

## Prepare confidence-change data and model helpers

In [ ]:
ROOT <- normalizePath("..", winslash = "/")
PATH <- file.path(ROOT, "data", "Psy_Case_Vignette_Data_Prolific_analysis.xlsx")
VIGNETTES <- c("A", "B", "C", "D", "E")
ARM_CODES <- c("4", "1", "2", "3")
ARM_LABELS <- c("Control", "ChatGPT", "Copilot", "Perplexity")
ARM_REF <- "Control"
TRANSITION_LEVELS <- c("right_right", "wrong_right", "wrong_wrong", "right_wrong")

df <- read_excel(PATH)

make_long_confidence <- function(data, pre_correct_cols, post_correct_cols, pre_conf_cols, post_conf_cols) {
  bind_rows(lapply(seq_along(VIGNETTES), function(i) {
    data %>%
      transmute(
        prolific_id = as.factor(prolific_id),
        vignette = factor(VIGNETTES[i], levels = VIGNETTES),
        study_arm = as.character(study_arm),
        Age = Age,
        Sex = as.character(Sex),
        pre_correct = .data[[pre_correct_cols[i]]],
        post_correct = .data[[post_correct_cols[i]]],
        pre_confidence = .data[[pre_conf_cols[i]]],
        post_confidence = .data[[post_conf_cols[i]]]
      ) %>%
      mutate(
        delta_confidence = post_confidence - pre_confidence,
        transition = factor(
          case_when(
            is.na(pre_correct) | is.na(post_correct) ~ NA_character_,
            pre_correct == 0 & post_correct == 1 ~ "wrong_right",
            pre_correct == 1 & post_correct == 1 ~ "right_right",
            pre_correct == 0 & post_correct == 0 ~ "wrong_wrong",
            pre_correct == 1 & post_correct == 0 ~ "right_wrong",
            TRUE ~ NA_character_
          ),
          levels = TRANSITION_LEVELS
        )
      )
  })) %>%
    filter(
      !is.na(delta_confidence), !is.na(transition), !is.na(study_arm),
      !is.na(prolific_id), !is.na(Age), !is.na(Sex)
    ) %>%
    mutate(
      study_arm = factor(study_arm, levels = ARM_CODES, labels = ARM_LABELS),
      study_arm = relevel(study_arm, ref = ARM_REF),
      Sex = relevel(factor(Sex), ref = "Female")
    )
}

coef_p_value <- function(coef_mat, row_name) {
  if ("Pr(>|t|)" %in% colnames(coef_mat)) return(coef_mat[row_name, "Pr(>|t|)"])
  if ("Pr(>|z|)" %in% colnames(coef_mat)) return(coef_mat[row_name, "Pr(>|z|)"])
  2 * pnorm(-abs(coef_mat[row_name, "t value"]))
}

print_lmer_terms <- function(model, label) {
  cat("\n", label, "\n", sep = "")
  cat(
    "N participants:", n_distinct(model@frame$prolific_id),
    "; N observations:", nrow(model@frame), "\n\n"
  )
  print(summary(model))
  coef_mat <- coef(summary(model))
  int_rows <- grep(":", rownames(coef_mat), value = TRUE)
  int_rows <- int_rows[grep("transition.*study_arm|study_arm.*transition", int_rows)]
  cat("\ntransition × study_arm interactions (vs control at right_right reference):\n")
  for (nm in int_rows) {
    b <- coef_mat[nm, "Estimate"]
    se <- coef_mat[nm, "Std. Error"]
    p <- coef_p_value(coef_mat, nm)
    cat(sprintf(
      "  %s: estimate = %.3f [%.3f, %.3f], p = %.4f\n",
      nm, b, b - 1.96 * se, b + 1.96 * se, p
    ))
  }
  invisible(model)
}

print_arm_vs_control_by_transition <- function(model, label) {
  cat("\n", label, "\n", sep = "")
  cat("LLM arm − Control on Δconfidence within each transition\n\n")
  emm <- emmeans(model, ~ study_arm | transition)
  cat("Estimated marginal means:\n")
  print(summary(emm, infer = c(TRUE, TRUE)))
  cat("\nContrasts vs Control:\n")
  ctr <- contrast(emm, "trt.vs.ctrl", adjust = "none")
  print(summary(ctr, infer = c(TRUE, TRUE), adjust = "none"))
  invisible(ctr)
}

PRE_CONDITION_COLS <- paste0("vig_", VIGNETTES, "_pre_condition_correct")
POST_CONDITION_COLS <- paste0("vig_", VIGNETTES, "_post_condition_correct")
PRE_NEXTSTEP_COLS <- paste0("vig_", VIGNETTES, "_pre_nextstep_correct")
POST_NEXTSTEP_COLS <- paste0("vig_", VIGNETTES, "_post_nextstep_correct")
PRE_CONF_COLS <- paste0("vig_", VIGNETTES, "_pre_confidence")
POST_CONF_COLS <- paste0("vig_", VIGNETTES, "_post_confidence")

long_condition <- make_long_confidence(
  df, PRE_CONDITION_COLS, POST_CONDITION_COLS, PRE_CONF_COLS, POST_CONF_COLS
)
long_triage <- make_long_confidence(
  df, PRE_NEXTSTEP_COLS, POST_NEXTSTEP_COLS, PRE_CONF_COLS, POST_CONF_COLS
)

cat("long_condition:", nrow(long_condition), "rows,", n_distinct(long_condition$prolific_id), "participants\n")
cat("long_triage:   ", nrow(long_triage), "rows,", n_distinct(long_triage$prolific_id), "participants\n")
cat("study_arm levels:", paste(levels(long_condition$study_arm), collapse = ", "), "\n")
cat("transition levels:", paste(levels(long_condition$transition), collapse = ", "), "\n")

### Fit the unadjusted condition model

In [ ]:
condition_unadj <- lmer(
  delta_confidence ~ transition * study_arm + (1 | prolific_id) + (1 | vignette),
  data = long_condition,
  REML = TRUE
)

print_lmer_terms(condition_unadj, "Model 1 — Confidence change, condition diagnosis (unadjusted)")

### Fit the unadjusted triage model

In [ ]:
triage_unadj <- lmer(
  delta_confidence ~ transition * study_arm + (1 | prolific_id) + (1 | vignette),
  data = long_triage,
  REML = TRUE
)

print_lmer_terms(triage_unadj, "Model 2 — Confidence change, triage / next step (unadjusted)")

## Estimate unadjusted LLM–control contrasts by trajectory

In [ ]:
condition_contrasts <- print_arm_vs_control_by_transition(
  condition_unadj,
  "Model 1 — Condition diagnosis"
)

triage_contrasts <- print_arm_vs_control_by_transition(
  triage_unadj,
  "Model 2 — Triage / next step"
)

## Prepare covariate-adjusted data

Use complete cases for age, sex, AI-use frequency, and AI literacy.

In [ ]:
ADJ_COVARIATES <- c("Age", "Sex", "ai_frequency", "ai_literacy_sum")
AILI_ITEMS <- paste0("aili_", sprintf("%02d", 1:12))
REVERSE_AILI_ITEMS <- c("aili_02", "aili_05", "aili_11")

recode_aili_col <- function(col_name) {
  values <- as.numeric(df[[col_name]])
  if (col_name %in% REVERSE_AILI_ITEMS) {
    values <- 8 - values
  }
  values
}

aili_recoded <- as.data.frame(lapply(AILI_ITEMS, recode_aili_col), stringsAsFactors = FALSE)
names(aili_recoded) <- AILI_ITEMS
df$ai_literacy_sum <- apply(aili_recoded, 1, function(row) {
  if (any(is.na(row))) NA_real_ else sum(row)
})

ai_covariates <- df %>%
  transmute(
    prolific_id = as.factor(prolific_id),
    ai_frequency = as.numeric(ai_frequency),
    ai_literacy_sum = as.numeric(ai_literacy_sum)
  )

add_ai_covariates <- function(long_data) {
  long_data %>%
    select(-any_of(c("ai_frequency", "ai_literacy_sum"))) %>%
    left_join(ai_covariates, by = "prolific_id")
}

print_transition_counts <- function(long_data, label) {
  cat("\n", label, "\n", sep = "")
  long_data %>%
    group_by(transition) %>%
    summarise(
      n_observations = n(),
      n_participants = n_distinct(prolific_id),
      .groups = "drop"
    ) %>%
    arrange(transition) %>%
    { print(as.data.frame(.), row.names = FALSE) }
}

print_transition_arm_counts <- function(long_data, label) {
  cat("\n", label, "— n observations by arm × transition\n", sep = "")
  long_data %>%
    count(study_arm, transition, name = "n") %>%
    arrange(study_arm, transition) %>%
    { print(as.data.frame(.), row.names = FALSE) }
}

long_condition_adj <- add_ai_covariates(long_condition) %>%
  filter(!is.na(ai_frequency), !is.na(ai_literacy_sum))
long_triage_adj <- add_ai_covariates(long_triage) %>%
  filter(!is.na(ai_frequency), !is.na(ai_literacy_sum))

cat("long_condition_adj:", nrow(long_condition_adj), "rows,", n_distinct(long_condition_adj$prolific_id), "participants\n")
cat("long_triage_adj:   ", nrow(long_triage_adj), "rows,", n_distinct(long_triage_adj$prolific_id), "participants\n")

print_transition_counts(long_condition_adj, "Condition diagnosis (adjusted sample)")
print_transition_arm_counts(long_condition_adj, "Condition diagnosis (adjusted sample)")

print_transition_counts(long_triage_adj, "Triage / next step (adjusted sample)")
print_transition_arm_counts(long_triage_adj, "Triage / next step (adjusted sample)")

### Fit the adjusted condition model

In [ ]:
condition_adj <- lmer(
  delta_confidence ~ transition * study_arm + Age + Sex + ai_frequency + ai_literacy_sum +
    (1 | prolific_id) + (1 | vignette),
  data = long_condition_adj,
  REML = TRUE
)

print_lmer_terms(
  condition_adj,
  "Model 3 — Confidence change, condition diagnosis (adjusted for Age, Sex, AI frequency, AI literacy)"
)

### Fit the adjusted triage model

In [ ]:
triage_adj <- lmer(
  delta_confidence ~ transition * study_arm + Age + Sex + ai_frequency + ai_literacy_sum +
    (1 | prolific_id) + (1 | vignette),
  data = long_triage_adj,
  REML = TRUE
)

print_lmer_terms(
  triage_adj,
  "Model 4 — Confidence change, triage / next step (adjusted for Age, Sex, AI frequency, AI literacy)"
)

## Estimate adjusted LLM–control contrasts by trajectory

In [ ]:
condition_adj_contrasts <- print_arm_vs_control_by_transition(
  condition_adj,
  "Model 3 — Condition diagnosis (adjusted)"
)

triage_adj_contrasts <- print_arm_vs_control_by_transition(
  triage_adj,
  "Model 4 — Triage / next step (adjusted)"
)

## Prepare transcript-verification sensitivity samples

SA1 excludes LLM participants with any category-4 consultation. SA2 additionally removes category-2 or category-3 LLM consultations; control observations remain unchanged.

In [ ]:
LLM_ARM_CODES <- c("1", "2", "3")
VALIDATION_COLS <- paste0("vig_", VIGNETTES, "_post_transcript_validation")

excluded_prolific_ids <- function(data, validation_categories) {
  sub <- data %>% filter(as.character(study_arm) %in% LLM_ARM_CODES)
  vals <- sub %>% select(all_of(VALIDATION_COLS))
  has_flag <- apply(vals, 1, function(row) {
    any(as.numeric(row) %in% validation_categories, na.rm = TRUE)
  })
  sub$prolific_id[has_flag]
}

filter_by_validation <- function(data, validation_categories, label) {
  exclude_ids <- excluded_prolific_ids(data, validation_categories)
  cat(
    label, "— excluding", length(exclude_ids),
    "LLM participant(s) with validation in {",
    paste(validation_categories, collapse = ", "), "}\n",
    sep = ""
  )
  filtered <- data %>% filter(!prolific_id %in% exclude_ids)
  cat("Remaining participants:", nrow(filtered), "\n\n")
  filtered
}

vignette_validation_long <- function(data) {
  bind_rows(lapply(seq_along(VIGNETTES), function(i) {
    data %>%
      transmute(
        prolific_id = as.factor(prolific_id),
        vignette = factor(VIGNETTES[i], levels = VIGNETTES),
        validation = as.numeric(.data[[VALIDATION_COLS[i]]])
      )
  }))
}

drop_validation_vignettes <- function(long_data, data_wide, validation_categories, label) {
  val_long <- vignette_validation_long(data_wide)
  merged <- long_data %>% left_join(val_long, by = c("prolific_id", "vignette"))
  n_before <- nrow(merged)
  filtered <- merged %>%
    filter(is.na(validation) | !(validation %in% validation_categories)) %>%
    select(-validation)
  cat(
    label, "— excluding", n_before - nrow(filtered),
    "observation(s) from vignettes with validation in {",
    paste(validation_categories, collapse = ", "), "}\n\n",
    sep = ""
  )
  filtered
}

fit_unadjusted_confidence <- function(long_data, label) {
  model <- lmer(
    delta_confidence ~ transition * study_arm + (1 | prolific_id) + (1 | vignette),
    data = long_data,
    REML = TRUE
  )
  print_lmer_terms(model, label)
  invisible(model)
}

df_sa1 <- filter_by_validation(df, 4, "SA1")
df_sa2_participants <- filter_by_validation(df, 4, "SA2 participant exclusion")

long_condition_sa1 <- make_long_confidence(
  df_sa1, PRE_CONDITION_COLS, POST_CONDITION_COLS, PRE_CONF_COLS, POST_CONF_COLS
)
long_triage_sa1 <- make_long_confidence(
  df_sa1, PRE_NEXTSTEP_COLS, POST_NEXTSTEP_COLS, PRE_CONF_COLS, POST_CONF_COLS
)

long_condition_sa2 <- make_long_confidence(
  df_sa2_participants, PRE_CONDITION_COLS, POST_CONDITION_COLS, PRE_CONF_COLS, POST_CONF_COLS
) %>%
  drop_validation_vignettes(df_sa2_participants, c(2, 3), "SA2 vignette exclusion (condition)")
long_triage_sa2 <- make_long_confidence(
  df_sa2_participants, PRE_NEXTSTEP_COLS, POST_NEXTSTEP_COLS, PRE_CONF_COLS, POST_CONF_COLS
) %>%
  drop_validation_vignettes(df_sa2_participants, c(2, 3), "SA2 vignette exclusion (triage)")

cat("SA1 long_condition:", nrow(long_condition_sa1), "rows,", n_distinct(long_condition_sa1$prolific_id), "participants\n")
cat("SA1 long_triage:   ", nrow(long_triage_sa1), "rows,", n_distinct(long_triage_sa1$prolific_id), "participants\n")
cat("SA2 long_condition:", nrow(long_condition_sa2), "rows,", n_distinct(long_condition_sa2$prolific_id), "participants\n")
cat("SA2 long_triage:   ", nrow(long_triage_sa2), "rows,", n_distinct(long_triage_sa2$prolific_id), "participants\n")

### Fit SA1 models

In [ ]:
cat(strrep("=", 88), "\n", sep = "")
cat("SA1 — Condition diagnosis (unadjusted; exclude validation category 4)\n")
cat(strrep("=", 88), "\n", sep = "")
condition_sa1 <- fit_unadjusted_confidence(
  long_condition_sa1,
  "SA1 — Confidence change, condition diagnosis (unadjusted; exclude validation category 4)"
)

cat(strrep("=", 88), "\n", sep = "")
cat("SA1 — Triage / next step (unadjusted; exclude validation category 4)\n")
cat(strrep("=", 88), "\n", sep = "")
triage_sa1 <- fit_unadjusted_confidence(
  long_triage_sa1,
  "SA1 — Confidence change, triage / next step (unadjusted; exclude validation category 4)"
)

### Fit SA2 models

In [ ]:
cat(strrep("=", 88), "\n", sep = "")
cat("SA2 — Condition diagnosis (unadjusted; exclude category 4 participants, categories 2–3 vignettes)\n")
cat(strrep("=", 88), "\n", sep = "")
condition_sa2 <- fit_unadjusted_confidence(
  long_condition_sa2,
  "SA2 — Confidence change, condition diagnosis (unadjusted; exclude category 4 participants, categories 2–3 vignettes)"
)

cat(strrep("=", 88), "\n", sep = "")
cat("SA2 — Triage / next step (unadjusted; exclude category 4 participants, categories 2–3 vignettes)\n")
cat(strrep("=", 88), "\n", sep = "")
triage_sa2 <- fit_unadjusted_confidence(
  long_triage_sa2,
  "SA2 — Confidence change, triage / next step (unadjusted; exclude category 4 participants, categories 2–3 vignettes)"
)

## Estimate sensitivity-analysis contrasts

### Estimate SA1 contrasts

In [ ]:
condition_sa1_contrasts <- print_arm_vs_control_by_transition(
  condition_sa1,
  "SA1 — Condition diagnosis"
)

triage_sa1_contrasts <- print_arm_vs_control_by_transition(
  triage_sa1,
  "SA1 — Triage / next step"
)

### Estimate SA2 contrasts

In [ ]:
condition_sa2_contrasts <- print_arm_vs_control_by_transition(
  condition_sa2,
  "SA2 — Condition diagnosis"
)

triage_sa2_contrasts <- print_arm_vs_control_by_transition(
  triage_sa2,
  "SA2 — Triage / next step"
)